# 第 67 课｜从字符级 RLVR 走向真实 Tokenizer、Chat Template 与模型接口

**从自己定义的字符 ID，走到可以接入真实预训练文本模型的输入、标签、结束和奖励契约。**

上一课在自编字符级 Transformer 上验证 token 级 RLVR、组优势和资源预算。本课不直接对数十亿参数模型训练：先把容易静默出错的**分词边界、对话模板、assistant 前缀、EOS/EOT、PAD、右填充、token log 概率**做成可测试接口，再提供显式启用的**本地 Hugging Face tokenizer 与 CausalLM 入口**。

默认实验完全离线，仅需要 Python 3.10+ 与已有 PyTorch 环境；不下载权重或额外包。可选真实 HF 路径需要用户**预先准备本地模型**并安装相容的 Transformers；该路径在本课制作环境中没有真实模型和 Transformers，因此**未声称已经实测预训练模型的质量、兼容性或 3060 显存**。

独立的 `.py` 和本 Notebook 互不依赖。**这是一堂迁移/接口审计课，不是LLM能力成绩或新的 RLVR 训练。**

## 0. 学习目标与顺序

完成后，你应能亲自检查五个问题：

1. Tokenizer 把文本变成什么 IDs？为什么把两段分别 encode 再拼接，不一定等于整体 encode？
2. Chat Template 的结构 token 是谁添加的？什么时候使用 `add_generation_prompt=True`，什么情况下不能加？
3. 为什么 SFT/DPO/RLVR 的答案标签要**与输入同位，统一移位一次**？为什么 EOS 与 PAD 不能混算？
4. 预算用完、实际生成 EOS、外部执行中断，分别能否视为正确/负例？
5. 在准备调用本地预训练模型前，如何检查模型目录、模板、停止ID、设备以及保守显存下界？

先跑第 1–9 节离线实验，再补三道独立练习。最后按需要开启本地真实 tokenizer / model 检查。官方 Chat Template 和生成规则可能随 Transformers 版本变化；本课固定来源并要求以实际模型配置为准。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第67课：真实Tokenizer、Chat Template与LLM接口迁移边界。

默认离线：标准库 + PyTorch；自编byte-level合并词元与真正因果Transformer。
可选本地HF tokenizer/model，懒加载transformers且local_files_only=True、trust_remote_code=False。
不下载权重、不运行远端代码；没有HF依赖/预训练权重时仍可完成全部实验。
默认实验是协议/掩码/停止/模型接口验证，不是预训练LLM性能、训练或RLVR成绩。
"""
from __future__ import annotations


import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import sys
import tempfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要现有PyTorch环境；本课默认不自动安装依赖") from exc

VERSION = "lesson67-real-tokenizer-boundaries-v1"
IGNORE = -100
SOURCES = {
    "chat_template": "https://huggingface.co/docs/transformers/v4.52.3/chat_templating",
    "chat_current": "https://huggingface.co/docs/transformers/chat_templating",
    "generation": "https://huggingface.co/docs/transformers/main_classes/text_generation",
    "local_load": "https://huggingface.co/docs/transformers/main_classes/model",
    "trl_grpo": "https://huggingface.co/docs/trl/grpo_trainer",
    "pytorch": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""


def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")


def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value


def strict_json(text: str):
    bounded_str(text, 8192, "JSON")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复JSON字段")
            result[key] = value
        return result
    def bad(_):
        raise LessonError("不接受NaN/Infinity")
    try:
        return json.loads(text, object_pairs_hook=pairs, parse_constant=bad)
    except (json.JSONDecodeError, RecursionError, UnicodeError) as exc:
        raise LessonError("JSON无法严格解析") from exc


def device_from_name(name: str):
    require(name in {"cpu", "cuda", "auto"}, "device应为cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    require(name != "cuda" or torch.cuda.is_available(), "CUDA不可用，不静默退回CPU")
    return torch.device(name)

In [2]:
torch.set_num_threads(1)
print('Python:',platform.python_version())
print('PyTorch:',torch.__version__)
print('CUDA可用:',torch.cuda.is_available())
print('默认无需Transformers，且不会访问模型仓库。')

Python: 3.13.5
PyTorch: 2.10.0+cpu
CUDA可用: False
默认无需Transformers，且不会访问模型仓库。


## 1. Tokenizer 的输出不是字符数组

真实子词 tokenizer 会合并不同长度的字节/字符片段。为了把边界问题做成可重复实验，本课实现一个**固定 UTF-8 byte-level 合并的教学 tokenizer**：普通文本可以按已知片段合并；结构控制 token 独占 ID，不从用户文本猜测。它不是真实 BPE、SentencePiece 或 Hugging Face FastTokenizer。

最小反例：

```text
encode('OK') → 一个合并 token
encode('O') + encode('K') → 两个单字节 token
```

两组 IDs 解码都可以得到 `OK`，但不是同一 token 序列。真实模型只接受自己训练时的 token ID 约定，不能拿课程的模拟 ID 去喂真实模型。

跨语言文本可能占多字节，即使一个输出 token 是 Unicode 字符的一部分，也不能直接把它当完整字符解析；本课的字节解码拒绝未完成的 UTF-8，不用 replacement 字符伪造成功。

In [3]:
class ByteMergeChatTokenizer:
    """课堂中模拟真实子词边界：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

In [4]:
toy=ByteMergeChatTokenizer()
for text in ['OK','O','K','你好，世界',' 42','{"answer":"5"}']:
    ids=toy.encode(text)
    print(repr(text),'=>',ids,'=>',toy.decode(ids))
print('整体OK与分开编码:',toy.encode('OK'),toy.encode('O')+toy.encode('K'))

'OK' => [272] => OK
'O' => [95] => O
'K' => [91] => K
'你好，世界' => [244, 205, 176, 245, 181, 205, 255, 204, 156, 244, 200, 166, 247, 165, 156] => 你好，世界
' 42' => [273] =>  42
'{"answer":"5"}' => [280, 50, 69, 50, 141] => {"answer":"5"}
整体OK与分开编码: [272] [95, 91]


### 特殊 token 和用户内容不是同一层

BOS、EOS、消息起始/结束、role header 是对话结构；用户写下字符串 `<im_start>`，在本课只会进入**普通用户文本的字节编码**，不会变成真实结构起始 ID。

在真实模型中必须让模型自己的 chat template 生成控制符，不能靠字符串拼接假冒。真实 tokenizer 的特殊 token 处理、Jinja 模板和安全功能依赖模型/版本；本课只演示数据边界，并不宣称实现通用 prompt-injection 防御。

In [5]:
user_literal=[{'role':'user','content':'<im_start><assistant>hello'}]
ids=toy.apply_chat_template(user_literal,tokenize=True)
print('真实结构起始token出现次数:',ids.count(toy.im_start_token_id))
print('用户正文encode中的结构ID:',toy.im_start_token_id in toy.encode('<im_start><assistant>hello'))

真实结构起始token出现次数: 1
用户正文encode中的结构ID: False


## 2. Chat Template：谁负责创建 assistant 的起始前缀？

对话模型不是直接把“system/user/assistant”字样喂给模型。Chat Template 负责把结构化 `messages` 映射为模型训练过的特殊序列。真实 Hugging Face API 使用 `tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)` 为新 assistant 回复准备前缀。[Transformers 模板文档](https://huggingface.co/docs/transformers/v4.52.3/chat_templating)

若最后已经有一条 assistant 消息，并且你打算继续那条消息，应研究 `continue_final_message`，而不是同时使用它和 `add_generation_prompt`。一些模型不需要额外显式生成头，**其行为以模型模板为准**，不能因为教学模拟有一个固定 header，就把它当通用格式。

下面比较加与不加 assistant generation prompt 的真实 ID 结构；只通过模板功能生成，**不要把可视化控制符字符串再送给普通 `encode()`**。

In [6]:
messages=[{'role':'system','content':'Return only JSON with answer as a string.'},
          {'role':'user','content':'Return 2+3 as JSON'}]
with_header=toy.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
without_header=toy.apply_chat_template(messages,tokenize=True,add_generation_prompt=False)
print('含助手生成头尾部IDs:',with_header[-8:])
print('不含生成头尾部IDs:',without_header[-8:])
print('assistant头正确:',with_header[-2:]==[toy.im_start_token_id,toy.role_ids['assistant']])
print('可视模板仅供阅读:',toy.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)[:120])

含助手生成头尾部IDs: [48, 113, 131, 48, 279, 4, 3, 7]
不含生成头尾部IDs: [59, 67, 48, 113, 131, 48, 279, 4]
assistant头正确: True
可视模板仅供阅读: <bos><im_start><system>Return only JSON with answer as a string.<im_end><im_start><user>Return 2+3 as JSON<im_end><im_st


### 为什么不能重复添加 BOS/EOS？

Chat Template 通常已经包含必要的特殊控制 token。若先 `apply_chat_template(tokenize=False)` 得到字符串，再调用默认会添加特殊 token 的编码函数，可能产生重复 BOS/EOS。HF 官方说明：这种两步流程一般要 `add_special_tokens=False`；直接 `apply_chat_template(tokenize=True)` 时不必自己补。[同一文档](https://huggingface.co/docs/transformers/v4.52.3/chat_templating)

本课的 `encode(text, add_special_tokens=True)` 故意模拟自动补 BOS 的行为。下面对比，**不是建议把教学 tokenizer 的可视字符串反编码当成真实模板**。

In [7]:
print('正文不自动补BOS:',toy.encode('OK'))
print('显式让编码器补BOS:',toy.encode('OK',add_special_tokens=True))
print('正确模板只包含一个BOS:',with_header.count(toy.bos_token_id))
print('误把可视模板交给普通正文编码：', toy.encode('<im_start>')[:8])

正文不自动补BOS: [272]
显式让编码器补BOS: [1, 272]
正确模板只包含一个BOS: 1
误把可视模板交给普通正文编码： [76, 121, 125, 111, 131, 132, 113, 130]


## 3. 用 token IDs 构造可训练的回答

本课固定一个已经构造好的 `assistant` 生成前缀，然后将**正文 token IDs + EOS ID**附到后面。这样得到一条明确的、自回归模型能够输入的合法 token 序列。它不意味着“分别 tokenization 然后拼接”与“整段文本重新 tokenization”一定给出相同 tokenization；在真实模型上，应检查完成边界和字符串重建契约。

标签与输入**同位**：

```text
input:  [BOS][system][user]...[assistant-prefix][answer-body][EOS]
labels:   -100  -100   -100        -100       body IDs  EOS ID
```

prompt 可作为上下文，但不直接计入回答损失。答案 EOS 是需要学习的终止动作，不能省掉；也不能把 EOS 后补齐当成回答。样本超出上下文时直接拒绝，**不静默裁掉回答或 EOS**。

In [8]:
def messages_for_task(question: str) -> list[dict]:
    bounded_str(question, 1500, "question")
    require(bool(question.strip()), "问题不能为空")
    return [{"role":"system","content":"Return only JSON with answer as a string."},
            {"role":"user","content":question}]


def build_supervised_sample(tokenizer: ByteMergeChatTokenizer, messages: list[dict],
                            response_text: str, *, max_tokens: int=512) -> dict:
    require(type(max_tokens) is int and 4 <= max_tokens <= 8192, "上下文上限不合法")
    bounded_str(response_text, 2048, "response")
    require(bool(response_text), "回答不能为空")
    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
    response_ids = tokenizer.encode(response_text,add_special_tokens=False)
    targets = response_ids + [tokenizer.eos_token_id]
    ids = prefix + targets
    require(len(ids) <= max_tokens, "样本超出上下文，不能静默截断EOS或回答")
    require(all(t >= tokenizer.byte_offset for t in response_ids), "回答不能包含结构控制token")
    labels = [IGNORE]*len(prefix)+targets
    return {"input_ids":ids,"labels":labels,"prompt_len":len(prefix),
            "completion_len":len(targets),"response_token_ids":targets,
            "prompt_digest":digest(prefix),"body_digest":digest(response_ids)}


def collate_right(samples: list[dict], pad_id: int, device="cpu") -> dict:
    require(type(samples) is list and bool(samples), "需要非空训练批次")
    require(type(pad_id) is int and pad_id >= 0, "pad ID非法")
    width = max(len(s["input_ids"]) for s in samples)
    ids = torch.full((len(samples),width),pad_id,dtype=torch.long,device=device)
    attention = torch.zeros_like(ids,dtype=torch.bool)
    labels = torch.full_like(ids,IGNORE)
    prompt_lens=[]; completion_lens=[]
    for i,s in enumerate(samples):
        row, lab = s["input_ids"],s["labels"]
        require(len(row)==len(lab) and len(row)>=2, "输入与标签不对齐")
        n = len(row);p=s["prompt_len"]
        require(type(p) is int and 1 <= p < n and s["completion_len"]==n-p,"回答边界错误")
        require(lab[:p]==[IGNORE]*p and lab[p:]==row[p:],"提示必须忽略，回答与EOS必须同位")
        ids[i,:n] = torch.tensor(row,device=device)
        labels[i,:n] = torch.tensor(lab,device=device)
        attention[i,:n] = True
        prompt_lens.append(p);completion_lens.append(n-p)
    return {"input_ids":ids,"attention_mask":attention,"labels":labels,
            "prompt_lens":prompt_lens,"completion_lens":completion_lens}


def shifted_logprobs(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    require(logits.ndim==3 and logits.shape[:2]==labels.shape and logits.shape[1]>=2,"logits/labels维度非法")
    require(logits.is_floating_point() and labels.dtype==torch.long,"logits需要浮点，labels需要long")
    require(bool(torch.isfinite(logits).all()),"logits不允许NaN/Infinity")
    require(bool((labels[:,0]==IGNORE).all()),"首位置没有前一预测状态，必须IGNORE")
    targets=labels[:,1:]
    mask=targets.ne(IGNORE)
    counts=mask.sum(-1)
    require(bool((counts>0).all()),"每条回答必须至少一个监督token")
    safe=targets.masked_fill(~mask,0)
    require(bool(((safe>=0)&(safe<logits.shape[-1])).all()),"词元标签越界")
    work = logits if logits.dtype in (torch.float32, torch.float64) else logits.float()
    logp=F.log_softmax(work[:,:-1],dim=-1)
    selected=logp.gather(-1,safe.unsqueeze(-1)).squeeze(-1)
    masked=selected.masked_fill(~mask,0)
    return {"per_token":masked,"mask":mask,"count":counts,
            "sum":masked.sum(-1),"per_token_mean":masked.sum(-1)/counts}

In [9]:
sample_ok=build_supervised_sample(toy,messages,'{"answer":"5"}')
sample_bad=build_supervised_sample(toy,messages,'{"answer":"6"}')
print('prompt_len:',sample_ok['prompt_len'],'answer+EOS长度:',sample_ok['completion_len'])
print('前缀的所有labels都为-100:',all(x==IGNORE for x in sample_ok['labels'][:sample_ok['prompt_len']]))
print('目标EOS ID:',sample_ok['response_token_ids'][-1])
print('目标回答:',toy.decode(sample_ok['response_token_ids'][:-1]))

prompt_len: 57 answer+EOS长度: 6
前缀的所有labels都为-100: True
目标EOS ID: 2
目标回答: {"answer":"5"}


## 4. 右补齐训练批次，不要与生成时左补齐的规则混淆

本课的微型模型使用**右补齐**训练输入；每行的真实 token 位置不变，PAD 的 attention mask 为假，label 为 −100。`attention_mask` 描述参与 attention 的 token，`labels` 决定计算损失的位置，二者用途不同。

真实批量自回归 `generate()` 常采用左补齐，使每条序列最后一个位置都是有效提示末尾；一些训练器要求 `padding_side='left'`。本课默认做**单样本真实模型生成**来回避此处多种位置 ID 与PAD行为差异，不能把右补齐训练的约定直接套到所有 batched generation。

In [10]:
batch=collate_right([sample_ok,sample_bad],toy.pad_token_id)
print('batch input shape:',tuple(batch['input_ids'].shape))
print('有效输入数量:',batch['attention_mask'].sum(-1).tolist())
print('有效回答目标数量:',(batch['labels'][:,1:]!=IGNORE).sum(-1).tolist())
print('EOS算作监督token:',int((batch['labels']==toy.eos_token_id).sum()))
print('padding处标签全是IGNORE:',bool((batch['labels'][~batch['attention_mask']]==IGNORE).all()))

batch input shape: (2, 63)
有效输入数量: [63, 63]
有效回答目标数量: [6, 6]
EOS算作监督token: 2
padding处标签全是IGNORE: True


## 5. 只移位一次，得到真实回答的 log 概率

完整回答序列概率是条件概率连乘，因此 log 概率要对有效目标求**和**：

\[\log\pi(y|x)=\sum_t\log\pi(y_t|x,y_{<t}).\]

本课的模型只输出 logits，不内部移位，所以计算器使用 `logits[:, :-1]` 预测 `labels[:, 1:]`。不要在构造样本时提前移位，也不要拿内部再移位一次的损失接口重复移位。

`-100` 不是词表 ID；先替换为安全 gather 编号，之后只对有效位置记分。FP16/BF16 使用 FP32 算 `log_softmax`，以改善数值稳定性。[PyTorch log_softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)

In [11]:
uniform=torch.zeros(2,batch['input_ids'].shape[1],toy.vocab_size,dtype=torch.float64,requires_grad=True)
scores=shifted_logprobs(uniform,batch['labels'])
print('有效target数量:',scores['count'].tolist())
print('均匀分布序列logp:',scores['sum'].detach().tolist())
print('手算:',[-int(n)*math.log(toy.vocab_size) for n in scores['count']])
scores['sum'].sum().backward()
print('提示无直接预测标签的位置梯度为0:',float(uniform.grad[:,:sample_ok['prompt_len']-1].abs().sum())==0)

有效target数量: [6, 6]
均匀分布序列logp: [-33.83012801600247, -33.83012801600247]
手算: [-33.830128016002476, -33.830128016002476]
提示无直接预测标签的位置梯度为0: True


## 6. EOS、EOT、PAD 和停止原因必须来自实际模型

有的模型以单个 `eos_token_id` 终止，有的模型 `generation_config.eos_token_id` 是一个列表，还可能使用与 tokenizer 默认 EOS 不同的回合结束符。**不能凭模型名字猜 ID；应读取实际 generation_config 与 Chat Template。**[Transformers 生成配置](https://huggingface.co/docs/transformers/main_classes/text_generation)

区分：

- `eos`：观测到指定终止 ID，且它是实际完成的最后一个 token。
- `budget_cap`：达到事先定义的最大输出长度，仍没有观测到终止 token。
- `interrupted`：运行错误或外部中断，没有可信的完整结果，**不能冒充零奖励负例**。

在本课的规定奖励任务中，缺少 EOS 的 cap 结果计零；但这是明确的任务规则，并不表示所有开放式问题的截断都等于错误。外部运行故障始终不作零奖励。

In [12]:
def normalize_stop_ids(eos_spec: int|list[int]|tuple[int,...]|None,
                       vocab_size: int|None=None) -> list[int]:
    if type(eos_spec) is int: values=[eos_spec]
    elif type(eos_spec) in (list,tuple) and bool(eos_spec): values=list(eos_spec)
    else: raise LessonError("未配置EOS/EOT时不能假定任何token是结束符")
    require(all(type(v) is int and v >=0 and (vocab_size is None or v < vocab_size)
                for v in values) and len(values)==len(set(values)),"停止ID重复或超出词表")
    return values


def audit_completion(generated_ids: list[int], stop_ids: list[int], limit: int,
                     *, interrupted: bool=False) -> dict:
    require(type(generated_ids) is list and all(type(i) is int and i >= 0 for i in generated_ids),
            "必须提供实际生成的token列表")
    require(type(limit) is int and limit>0, "预算必须是正整数")
    require(bool(stop_ids) and all(type(x) is int and x>=0 for x in stop_ids),"停止ID非法")
    require(len(generated_ids) <= limit,"实际输出越过预算")
    if interrupted:
        return {"state":"interrupted","completed":False,"stop_id":None,
                "body_ids":None,"generated_tokens":len(generated_ids),"grade_allowed":False}
    require(bool(generated_ids),"非中断但无输出：无法判断生成状态")
    stops=[i for i,v in enumerate(generated_ids) if v in stop_ids]
    require(not stops or stops==[len(generated_ids)-1],"结束符后还有token或重复结束符；须提供可信生成长度")
    if stops:
        return {"state":"eos","completed":True,"stop_id":generated_ids[-1],
                "body_ids":generated_ids[:-1],"generated_tokens":len(generated_ids),"grade_allowed":True}
    require(len(generated_ids)==limit,"未遇结束符但未耗尽预算，缺少可信停止元数据")
    return {"state":"budget_cap","completed":False,"stop_id":None,
            "body_ids":generated_ids[:],"generated_tokens":len(generated_ids),"grade_allowed":True}


def audit_batched_sequences(rows: list[list[int]], stop_ids: list[int],
                            prefix_lengths: list[int], max_new_tokens: int,
                            *, completion_lengths: list[int]|None=None,
                            pad_id: int|None=None) -> list[dict]:
    """要求实际长度；若pad==EOS，任何从填充推断EOS的方法都不安全。"""
    require(type(rows) is list and type(prefix_lengths) is list and
            len(rows)==len(prefix_lengths)>0,"批次结构错误")
    require(completion_lengths is not None,"batched generate必须提供可信完成长度/结束元数据")
    require(len(completion_lengths)==len(rows),"长度条数不匹配")
    output=[]
    for row,p,n in zip(rows,prefix_lengths,completion_lengths):
        require(type(p) is int and type(n) is int and 0<=p<len(row) and 1<=n<=max_new_tokens,
                "prefix/completion长度不合法")
        require(p+n<=len(row),"返回序列短于声明的完成长度")
        require(pad_id is None or all(x==pad_id for x in row[p+n:]),"尾部填充不是声明的PAD ID")
        output.append(audit_completion(row[p:p+n],stop_ids,max_new_tokens))
    return output


def grade_json_answer(body: str|None, state: str, expected: str) -> dict:
    """cap是任务预算结果0；interrupted是基础设施缺失，不能伪造负例。"""
    bounded_str(expected,100,"expected")
    require(re.fullmatch(r"[0-9]+",expected) is not None,"本课只使用非负整数真值")
    require(state in {"eos","budget_cap","interrupted"},"停止状态非法")
    if state=="interrupted":
        return {"reward":None,"eligible":False,"reason":"not_observed"}
    if state=="budget_cap":
        return {"reward":0.0,"eligible":True,"reason":"missing_stop"}
    bounded_str(body,8192,"body")
    try:
        data=strict_json(body)
        require(type(data) is dict and set(data)=={"answer"} and type(data["answer"]) is str,
                "回答JSON必须恰好有字符串answer字段")
    except LessonError:
        return {"reward":0.0,"eligible":True,"reason":"invalid_format"}
    passed=data["answer"]==expected
    return {"reward":float(passed),"eligible":True,
            "reason":"correct" if passed else "wrong_value"}

In [13]:
for tokens,limit,interrupt in [([21,2],3,False),([21,22,23],3,False),([21],3,True)]:
    print(tokens,'limit',limit,'interrupted',interrupt,'=>',
          audit_completion(tokens,[2],limit,interrupted=interrupt))
print('多个结束符例子：',audit_completion([21,4],[2,4],3))

[21, 2] limit 3 interrupted False => {'state': 'eos', 'completed': True, 'stop_id': 2, 'body_ids': [21], 'generated_tokens': 2, 'grade_allowed': True}
[21, 22, 23] limit 3 interrupted False => {'state': 'budget_cap', 'completed': False, 'stop_id': None, 'body_ids': [21, 22, 23], 'generated_tokens': 3, 'grade_allowed': True}
[21] limit 3 interrupted True => {'state': 'interrupted', 'completed': False, 'stop_id': None, 'body_ids': None, 'generated_tokens': 1, 'grade_allowed': False}
多个结束符例子： {'state': 'eos', 'completed': True, 'stop_id': 4, 'body_ids': [21], 'generated_tokens': 2, 'grade_allowed': True}


### 当 PAD 与 EOS 是同一个 ID，不能靠尾部数字猜测长度

一些 tokenizer 没有独立 pad_token，单条或批量生成时可能临时令 `pad_token_id=eos_token_id`。结束后序列里额外的 EOS 数字也可能只是**填充**。

因此，本课的批量审计函数要求提供真实完成长度或生成元数据，不能猜“最后一个 EOS 才是真的结束”。训练时也要保留明确的 `attention_mask`、`completion_mask` 与生成结束标记，不能从 token 值单独反推。

下面模拟同一 pad/eos ID 下的两条序列：一条真的 EOS，一条达到预算。

In [14]:
try:
    audit_batched_sequences([[9,21,2,2]],[2],[1],3,pad_id=2)
except LessonError as error:
    print('没有真实长度时拒绝:',error)
example=audit_batched_sequences([[9,21,2,2],[9,21,22,23]],[2],[1,1],3,
                                 completion_lengths=[2,3],pad_id=2)
print('带可信长度后:',[x['state'] for x in example])

没有真实长度时拒绝: batched generate必须提供可信完成长度/结束元数据
带可信长度后: ['eos', 'budget_cap']


## 7. 奖励器应先看停止状态，再解析格式

只检查字符串是否包含正确数字是不可靠的。本课任务要求回复**唯一一个 JSON 对象**，且恰好有字符串 `answer` 字段。

```json
{"answer":"5"}
```

完整 EOS 且数值字符串正确，奖励 1；其他合法完成但数值错误或格式错误奖励 0。JSON 字段重复、额外解释、数值与字符串类型混用、代码围栏都不能悄悄修复。

`budget_cap` 在本课任务中是 0；`interrupted` 返回 `None`，排除该训练记录。**这份离线规则不是人类偏好模型，也不是通用数学评测。**

In [15]:
for body,state in [('{"answer":"5"}','eos'),('{"answer":"6"}','eos'),
                   ('{"answer":"5","answer":"6"}','eos'),
                   ('{"answer":"5"}','budget_cap'),(None,'interrupted')]:
    print(repr(body),state,'=>',grade_json_answer(body,state,'5'))

'{"answer":"5"}' eos => {'reward': 1.0, 'eligible': True, 'reason': 'correct'}
'{"answer":"6"}' eos => {'reward': 0.0, 'eligible': True, 'reason': 'wrong_value'}
'{"answer":"5","answer":"6"}' eos => {'reward': 0.0, 'eligible': True, 'reason': 'invalid_format'}
'{"answer":"5"}' budget_cap => {'reward': 0.0, 'eligible': True, 'reason': 'missing_stop'}
None interrupted => {'reward': None, 'eligible': False, 'reason': 'not_observed'}


## 8. 用真正的因果 Transformer 连接接口，而不声称它已学会任务

下面的小模型仍是随机初始化：词嵌入、绝对位置、两层因果 Transformer、词表输出。模型实际前向、teacher forcing 和贪心/随机生成都会执行。

**它没有经过预训练，也不做本课 RLVR 更新。** 成绩与模型能力无关；真正要验证的是 logits 形状、未来token不可见、右补齐不影响有效位置输出，以及生成停止/评分能被严格记录。

本课 tokenizer 输出结构 ID 并不匹配任何真实 Hugging Face 预训练模型，所以不要混接 toy token IDs 与真实 checkpoint。

In [16]:
@dataclass(frozen=True)
class TinyConfig:
    vocab_size: int
    hidden: int=32
    heads: int=4
    layers: int=2
    max_context: int=256

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),"模型配置应为正整数")
        require(self.hidden % self.heads ==0 and self.max_context>=32,"注意力头或上下文配置不合法")


class CausalBlock(nn.Module):
    def __init__(self, cfg: TinyConfig):
        super().__init__()
        self.ln1=nn.LayerNorm(cfg.hidden)
        self.attn=nn.MultiheadAttention(cfg.hidden,cfg.heads,batch_first=True,dropout=0.0)
        self.ln2=nn.LayerNorm(cfg.hidden)
        self.ff=nn.Sequential(nn.Linear(cfg.hidden,4*cfg.hidden),nn.GELU(),
                              nn.Linear(4*cfg.hidden,cfg.hidden))

    def forward(self,x,pad_mask,causal_mask):
        y,_=self.attn(self.ln1(x),self.ln1(x),self.ln1(x),
                      key_padding_mask=pad_mask,attn_mask=causal_mask,need_weights=False)
        x=x+y
        return x+self.ff(self.ln2(x))


class TinyCausalLM(nn.Module):
    def __init__(self,cfg: TinyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.token=nn.Embedding(cfg.vocab_size,cfg.hidden)
        self.position=nn.Embedding(cfg.max_context,cfg.hidden)
        self.blocks=nn.ModuleList([CausalBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.hidden)
        self.head=nn.Linear(cfg.hidden,cfg.vocab_size,bias=False)

    def forward(self,input_ids: torch.Tensor,attention_mask: torch.Tensor) -> torch.Tensor:
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,"input_ids需要[B,T] long")
        b,t=input_ids.shape
        require(b>0 and 0<t<=self.cfg.max_context,"模型上下文超限")
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                "attention_mask形状/type错误")
        require(bool(attention_mask[:,0].all()),"课堂训练只支持右填充，拒绝左填充")
        require(not bool(((~attention_mask[:,:-1]) & attention_mask[:,1:]).any()),"不支持中间空洞")
        require(bool(((input_ids>=0)&(input_ids<self.cfg.vocab_size)).all()),"输入ID越界")
        arange=torch.arange(t,device=input_ids.device)
        x=self.token(input_ids)+self.position(arange)[None]
        causal_mask=torch.ones(t,t,dtype=torch.bool,device=input_ids.device).triu(1)
        for block in self.blocks:
            x=block(x,~attention_mask,causal_mask)
        return self.head(self.norm(x))


@torch.no_grad()
def tiny_generate(model: TinyCausalLM, prefix: list[int], tok: ByteMergeChatTokenizer,
                  max_new_tokens: int, *, greedy: bool=True, seed: int=67) -> dict:
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=64,"生成长度非法")
    require(type(prefix) is list and bool(prefix) and all(type(i) is int and 0<=i<tok.vocab_size for i in prefix),
            "prefix token不合法")
    require(len(prefix)+max_new_tokens<=model.cfg.max_context,"留给完成token的空间不足")
    device=next(model.parameters()).device
    generator=torch.Generator(device=device).manual_seed(seed)
    old=model.training;model.eval()
    generated=[]
    try:
        for _ in range(max_new_tokens):
            ids=torch.tensor([prefix+generated],dtype=torch.long,device=device)
            mask=torch.ones_like(ids,dtype=torch.bool)
            logits=model(ids,mask)[0,-1].clone()
            # 本例允许所有普通词元＋EOS，不允许模型输出结构控制字和保留空隙。
            eos_logit = logits[tok.eos_token_id].clone()
            logits[:tok.byte_offset] = -torch.inf
            logits[tok.eos_token_id] = eos_logit
            probs=F.softmax(logits,dim=0)
            token=int(probs.argmax()) if greedy else int(torch.multinomial(probs,1,generator=generator))
            generated.append(token)
            if token==tok.eos_token_id:break
    finally:
        model.train(old)
    return {"prefix_len":len(prefix),"generated_ids":generated,
            "audit":audit_completion(generated,[tok.eos_token_id],max_new_tokens),
            "generator":"tiny_random_causal_model_not_pretrained","sampled":not greedy}

In [17]:
torch.manual_seed(67)
tiny=TinyCausalLM(TinyConfig(toy.vocab_size)).eval()
with torch.no_grad():
    logits=tiny(batch['input_ids'],batch['attention_mask'])
    results=shifted_logprobs(logits,batch['labels'])
print('模型 logits shape:',list(logits.shape))
print('随机模型正确候选logp:',float(results['sum'][0]))
print('随机模型错误候选logp:',float(results['sum'][1]))
print('这两个数不是答案正确率，也不是 DPO/RLVR 训练结果。')

模型 logits shape: [2, 63, 281]
随机模型正确候选logp: -33.398216247558594
随机模型错误候选logp: -33.39098358154297
这两个数不是答案正确率，也不是 DPO/RLVR 训练结果。


### 自由生成与 teacher forcing 不是同一种评估

teacher forcing 给模型真实的上一 token；自由生成则把**自己刚生成的 token**送回模型，所以即使已知答案序列 logp 较高，也不能宣称它会被贪心选中。

本课把所有非回答用的结构 token 从生成动作集合屏蔽，但保留 EOS；这只是一项教学约定，不适用于所有模型。真实生成必须使用模型自己的生成配置和合法 ID 集合。

In [18]:
prefix=toy.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
generation=tiny_generate(tiny,prefix,toy,max_new_tokens=12,greedy=True)
print('原始新增token:',generation['generated_ids'])
print('停止审计:',generation['audit'])
try:
    print('只显示回答正文:',toy.decode(generation['audit']['body_ids']))
except LessonError as error:
    print('随机模型输出了不完整UTF8，按格式失败处理:',error)

原始新增token: [20, 127, 168, 223, 248, 214, 34, 205, 235, 76, 31, 198]
停止审计: {'state': 'budget_cap', 'completed': False, 'stop_id': None, 'body_ids': [20, 127, 168, 223, 248, 214, 34, 205, 235, 76, 31, 198], 'generated_tokens': 12, 'grade_allowed': True}
随机模型输出了不完整UTF8，按格式失败处理: 不完整UTF-8回答；不要用replacement字符伪装成功


## 9. 从字符串到 GPU 预算：先做必要条件，不要当充分保证

粗略下界：

\[M_{weights}\ge \lceil P\cdot b/8\rceil,\qquad
  M_{KV}\ge 2\,L\,H_{kv}\,d_{head}\,B\,T\,s_{kv}.\]

这里 `P` 为参数量，`b` 为权重存储位数，L 为层数，$H_{kv}$ 为KV头数，$d_{head}$ 为单头维度，B 为 batch，T 为缓存序列长度，$s_{kv}$ 为每元素字节数。**因子2同时包含 Key 与 Value**。

没计量化元数据、激活、工作区、logits、框架分配器、额外参考/奖励/优化器权重，也没有测设备。**如果下界超过显存，方案行不通；下界低于显存，不保证能运行。** 以下 12GiB 只是显式的假设预算，不暗示任何用户显卡的具体显存。

In [19]:
def memory_lower_bound(parameter_count: int, bits: int, layers: int, kv_heads: int,
                       head_dim: int, batch: int, context_tokens: int,
                       *, kv_bytes: int=2, vram_bytes: int|None=None) -> dict:
    require(all(type(v) is int and v>0 for v in
                [parameter_count,bits,layers,kv_heads,head_dim,batch,context_tokens,kv_bytes]),
            "资源预算参数必须为正整数")
    require(bits in {4,8,16,32},"教学权重位宽只接受4/8/16/32")
    weights = math.ceil(parameter_count * bits / 8)
    kv = 2 * layers * kv_heads * head_dim * batch * context_tokens * kv_bytes
    lower = weights + kv
    if vram_bytes is not None: require(type(vram_bytes) is int and vram_bytes>0,"VRAM预算非法")
    return {"weights_lower_bytes":weights,"kv_lower_bytes":kv,
            "sum_lower_bytes":lower,
            "specified_vram_bytes":vram_bytes,
            "already_exceeds_budget":None if vram_bytes is None else lower>vram_bytes,
            "upper_bound_or_fit_guarantee":False,
            "omitted":"quantization metadata, activations, logits, workspaces, allocator, OS, optimizer, duplicated models"}


def optional_cuda_info() -> dict:
    if not torch.cuda.is_available():return {"available":False,"measured":False}
    free,total=torch.cuda.mem_get_info()
    return {"available":True,"free_bytes":int(free),"total_bytes":int(total),
            "measured":True,"not_model_fit_guarantee":True}

In [20]:
example_budget=memory_lower_bound(parameter_count=1_500_000_000,bits=4,layers=28,
      kv_heads=4,head_dim=128,batch=1,context_tokens=1024,kv_bytes=2,
      vram_bytes=12*1024**3)
print(json.dumps(example_budget,ensure_ascii=False,indent=2))
print('是否有CUDA可测:',optional_cuda_info())

{
  "weights_lower_bytes": 750000000,
  "kv_lower_bytes": 58720256,
  "sum_lower_bytes": 808720256,
  "specified_vram_bytes": 12884901888,
  "already_exceeds_budget": false,
  "upper_bound_or_fit_guarantee": false,
  "omitted": "quantization metadata, activations, logits, workspaces, allocator, OS, optimizer, duplicated models"
}
是否有CUDA可测: {'available': False, 'measured': False}


## 10. 连接本地真实 HF 模型：必须显式授权执行

完整文件提供两个单独入口：

- `--inspect-local /实际本地模型目录`：只加载 tokenizer，调用它自身的 `apply_chat_template`，记录 prompt IDs、EOS/PAD；**不加载权重**。
- `--local-model /实际本地模型目录`：要求预先放好的**本地纯文本 decoder-only checkpoint**，然后完成一个问题的 teacher-forced log 概率和单样本贪心生成。

二者均要求真实本地 `config.json`，`local_files_only=True`、`trust_remote_code=False`；实际模型权重模式还要求 safetensors。**不给仓库ID、不使用远程下载，也不自动安装包。** 某些模型需要自定义代码、不同模板、额外处理器或更新版 Transformers，当前教学入口会拒绝，不试图偷换为通用支持。

可参考实际环境里的 `python -m pip install transformers safetensors`，但请固定适合 checkpoint 的版本。不同 Transformers 大版本可能变更 API，所以本课用清晰的适配层隔离风险。

In [21]:
def local_model_directory(value: str|Path) -> Path:
    raw=Path(value).expanduser()
    require(not raw.is_symlink(), "模型目录不能是软链接")
    p=raw.resolve(strict=True)
    require(p.is_dir() and (p/"config.json").is_file(),
            "只允许包含config.json的真实本地模型目录；不能传远程仓库ID")
    require(not p.is_symlink(),"模型目录不能是软链接")
    return p


def _import_transformers():
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
    except ImportError as exc:
        raise LessonError("本地HF模式需要额外安装兼容版本的transformers；默认课堂无需它") from exc
    return AutoTokenizer, AutoModelForCausalLM


def hf_message_prefix(tokenizer, messages: list[dict]) -> list[int]:
    require(getattr(tokenizer,"chat_template",None) is not None,
            "模型Tokenizer没有chat_template，不能猜测它的对话格式")
    result=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
    if isinstance(result,torch.Tensor):result=result.tolist()
    require(type(result) is list and bool(result) and all(type(x) is int for x in result),
            "真实chat_template必须生成一维token IDs")
    return result


def inspect_local_tokenizer(path: str|Path) -> dict:
    directory=local_model_directory(path)
    AutoTokenizer,_ = _import_transformers()
    tokenizer=AutoTokenizer.from_pretrained(str(directory),local_files_only=True,trust_remote_code=False)
    messages=messages_for_task("Return the sum of 2 and 3 as JSON.")
    prefix=hf_message_prefix(tokenizer,messages)
    end_value=getattr(tokenizer,"eos_token_id",None)
    model_spec={"model_directory":str(directory),"tokenizer_class":tokenizer.__class__.__name__,
                "has_chat_template":True,"prompt_ids_count":len(prefix),
                "prompt_sha256":digest(prefix),"pad_token_id":getattr(tokenizer,"pad_token_id",None),
                "tokenizer_eos":end_value,"vocab_size":len(tokenizer),
                "local_files_only":True,"trust_remote_code":False,"weights_loaded":False}
    return model_spec


def run_local_hf(path: str|Path, device_name: str, max_new_tokens: int=32) -> dict:
    """可选本地单样本、文本模型；不提供预训练模型安装、训练或权重兼容承诺。"""
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=256,"生成长度非法")
    directory=local_model_directory(path)
    AutoTokenizer,AutoModelForCausalLM=_import_transformers()
    device=device_from_name(device_name)
    tokenizer=AutoTokenizer.from_pretrained(str(directory),local_files_only=True,trust_remote_code=False)
    messages=messages_for_task("Return the sum of 2 and 3 as JSON.")
    prefix=hf_message_prefix(tokenizer,messages)
    # 直接生成完整回答token，以保证训练目标与这份prefix契约一致。
    response_text='{"answer":"5"}'
    answer_ids=tokenizer.encode(response_text,add_special_tokens=False)
    require(bool(answer_ids),"真实tokenizer不能把回答变成空列表")
    # 停止ID来自实际generation_config或tokenizer；支持多个EOS/EOT；不臆造模板结束符。
    dtype=torch.float16 if device.type=='cuda' else torch.float32
    model=AutoModelForCausalLM.from_pretrained(str(directory),local_files_only=True,
            trust_remote_code=False,dtype=dtype,use_safetensors=True)
    require(not bool(getattr(model.config,"is_encoder_decoder",False)),"只支持decoder-only纯文本CausalLM")
    model.to(device).eval()
    generated_eos=getattr(getattr(model,"generation_config",None),"eos_token_id",None)
    if generated_eos is None:generated_eos=getattr(tokenizer,"eos_token_id",None)
    eos_ids=normalize_stop_ids(generated_eos,len(tokenizer))
    eos_id=eos_ids[0]
    # 不能保证所有模型使用首个stop作为唯一训练终止符：报告会暴露实际配置。
    sample=prefix+answer_ids+[eos_id]
    max_positions=getattr(model.config,"max_position_embeddings",None)
    if type(max_positions) is int and max_positions > 0:
        require(max(len(sample),len(prefix)+max_new_tokens) <= max_positions,
                "请求长度超过已声明位置上限")
    ids=torch.tensor([sample],dtype=torch.long,device=device)
    attention=torch.ones_like(ids,dtype=torch.long)
    labels=torch.tensor([[IGNORE]*len(prefix)+answer_ids+[eos_id]],dtype=torch.long,device=device)
    with torch.no_grad():
        out=model(input_ids=ids,attention_mask=attention)
        probabilities=shifted_logprobs(out.logits,labels)
    prompt=torch.tensor([prefix],dtype=torch.long,device=device)
    prompt_mask=torch.ones_like(prompt,dtype=torch.long)
    pad_id=getattr(tokenizer,"pad_token_id",None)
    if pad_id is None: pad_id=eos_id  # 单个请求，无任何填充；不适用批量训练标签推断。
    with torch.no_grad():
        generated=model.generate(input_ids=prompt,attention_mask=prompt_mask,
                                 do_sample=False,max_new_tokens=max_new_tokens,
                                 eos_token_id=eos_ids,pad_token_id=pad_id,
                                 return_dict_in_generate=True,output_scores=False)
    continuation=generated.sequences[0,len(prefix):].tolist()
    audit=audit_completion(continuation,eos_ids,max_new_tokens)
    # 单样本只依赖generation给出的序列：无推断批处理尾PAD的逻辑。
    body=tokenizer.decode(audit["body_ids"],skip_special_tokens=True) if audit["body_ids"] is not None else None
    grade=grade_json_answer(body,audit["state"],"5")
    return {"mode":"real_local_HF_greedy_inference_not_training",
            "model_directory":str(directory),"device":str(device),
            "local_files_only":True,"trust_remote_code":False,"safetensors_required":True,"model_loaded":True,
            "tokenizer_class":tokenizer.__class__.__name__,"model_class":model.__class__.__name__,
            "stop_ids":eos_ids,"effective_pad_id":pad_id,
            "pad_alias_eos":getattr(tokenizer,"pad_token_id",None) is None,
            "prefix_ids_count":len(prefix),"chosen_ids_count":len(answer_ids)+1,
            "reference_answer_logp":float(probabilities["sum"][0]),
            "reference_answer_token_count":int(probabilities["count"][0]),
            "raw_generated_ids":continuation,"completion_audit":audit,
            "generation_text":body,"grade":grade,
            "limitations":["one_prompt_no_model_benchmark","not_a_training_run",
                           "local_version_and_model_template_specific","eos_stop_ids_from_model_config"]}

In [22]:
RUN_LOCAL_TOKENIZER = False
RUN_LOCAL_MODEL = False
LOCAL_MODEL_DIR = ''  # 改成你自己已经下载的真实本地checkpoint目录。
LOCAL_DEVICE = 'cpu'

if RUN_LOCAL_TOKENIZER:
    print(json.dumps(inspect_local_tokenizer(LOCAL_MODEL_DIR),ensure_ascii=False,indent=2))
else:
    print('默认不加载外部tokenizer或模型。')

if RUN_LOCAL_MODEL:
    real_result=run_local_hf(LOCAL_MODEL_DIR,LOCAL_DEVICE,32)
    print(json.dumps(real_result,ensure_ascii=False,indent=2))
else:
    print('真实HF推理路径未执行；不虚构模型成绩或显存数据。')

默认不加载外部tokenizer或模型。
真实HF推理路径未执行；不虚构模型成绩或显存数据。


### 本地真实模型接口做了哪些限制？

只尝试一个文本问句、一个 `apply_chat_template(... tokenize=True, add_generation_prompt=True)` 前缀；不做多轮历史、工具调用、图片、视频、并发生成或训练。

参考目标按当前前缀拼接指定答案 token 和配置中的第一个结束 ID；这是一个**明确选择的合法 ID 序列**，不保证和重新编码完整对话的 IDs 相同，也不保证它是该模型最优的 assistant turn 终止 token。真实 EOS/EOT 列表全部保存，生成以模型配置为准。

单样本生成没有填充；pad缺失时仅为 `generate()` 提供EOS别名，同时记录这一选择。不能以此推断训练批次中 EOS 与 padding 的关系。**只有真实本地模型成功加载，才会标注真实模型已加载；默认离线示例始终为 toy 模型。**

## 11. 默认完整实验报告（不依赖真实模型）

下面综合运行模板、成对样本、log概率、随机小模型自回归、停止/评分、显存下界，保存独立时间戳目录：

```text
lesson_67_outputs/run_.../
    contract.json
    experiment.json
    checks.json
```

这不是“随机模型经过RLVR训练”的报告。`teacher_forced` 只记录给定 token 序列的条件概率；`generation` 记录实际产生的 ID、EOS/预算状态和本课格式奖励。

In [23]:
def demonstration() -> dict:
    torch.set_num_threads(1)
    torch.manual_seed(67)
    tok=ByteMergeChatTokenizer()
    messages=messages_for_task("Return 2+3 as JSON")
    prefix=tok.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
    with_user=tok.apply_chat_template(messages,tokenize=True,add_generation_prompt=False)
    chosen=build_supervised_sample(tok,messages,'{"answer":"5"}')
    rejected=build_supervised_sample(tok,messages,'{"answer":"6"}')
    batch=collate_right([chosen,rejected],tok.pad_token_id)
    model=TinyCausalLM(TinyConfig(tok.vocab_size)).eval()
    with torch.no_grad():
        probs=shifted_logprobs(model(batch["input_ids"],batch["attention_mask"]),batch["labels"])
    generated=tiny_generate(model,prefix,tok,24,greedy=True)
    audit=generated["audit"]
    text=None;grade={"reward":None,"eligible":False,"reason":"not_observed"}
    if audit["body_ids"] is not None:
        try:
            text=tok.decode(audit["body_ids"],skip_special_tokens=True)
            grade=grade_json_answer(text,audit["state"],"5")
        except LessonError:
            grade={"reward":0.0,"eligible":True,"reason":"invalid_utf8_or_format"}
    budget=memory_lower_bound(1_500_000_000,4,28,4,128,1,1024,vram_bytes=12*1024**3)
    return {"version":VERSION,"mode":"toy_tokenizer_random_tiny_causal_lm",
            "transformers_required":False,"model_pretrained":False,
            "tokenizer_contract":tok.contract(),"template":{
                "prefix_length":len(prefix),"without_assistant_header_length":len(with_user),
                "has_generation_header":prefix[-2:]==[tok.im_start_token_id,tok.role_ids["assistant"]],
                "prefix_sha256":digest(prefix),"example_single_text_merge":{
                    "whole_OK":tok.encode("OK"),"split_O_K":tok.encode("O")+tok.encode("K")}},
            "batch": {"shape":list(batch["input_ids"].shape),
                       "prompt_lens":batch["prompt_lens"],"completion_lens":batch["completion_lens"],
                       "answer_tokens_include_eos":True,
                       "prompt_and_pad_are_ignored":True},
            "teacher_forced": {"chosen_logp":float(probs["sum"][0]),
                               "rejected_logp":float(probs["sum"][1]),
                               "valid_targets":probs["count"].tolist()},
            "generation":{"generated_ids":generated["generated_ids"],
                          "audit":audit,"decoded_body":text,"grade":grade},
            "memory_lower_bound_example":budget,"cuda":optional_cuda_info(),
            "boundaries":["no_pretrained_model_or_real_HF_loading_in_default",
                           "toy_chat_template_is_not_any_real_model_template",
                           "no_RLVR_training_no_claimed_model_accuracy",
                           "HF_real_model_requires_explicit_local_files_and_transformers"]}

In [24]:
offline=demonstration()
print('默认运行模式:',offline['mode'])
print('有assistant生成头:',offline['template']['has_generation_header'])
print('Batch shape:',offline['batch']['shape'])
print('回答监督目标数:',offline['teacher_forced']['valid_targets'])
print('随机生成状态:',offline['generation']['audit']['state'])
print('随机生成奖励:',offline['generation']['grade'])

默认运行模式: toy_tokenizer_random_tiny_causal_lm
有assistant生成头: True
Batch shape: [2, 63]
回答监督目标数: [6, 6]
随机生成状态: budget_cap
随机生成奖励: {'reward': 0.0, 'eligible': True, 'reason': 'invalid_utf8_or_format'}


## 12. 独立练习：按顺序完成，再看参考答案

**练习 A：SFT / DPO / RLVR 的同位标签**

实现 `labels_ex(prefix, completion, eos)`，返回提示位置全为 `-100`、完成位置保留真实 token、末尾只附加一个 EOS。不能原地修改输入。

**练习 B：停止分类**

实现 `stop_state_ex(tokens, eos_ids, limit)`：若最后一个 ID 是任意允许 EOS/EOT，返回 `eos`；恰好耗尽预算且无结束 ID 为 `budget_cap`；中间出现结束、超预算或提前无缘由停止，都要拒绝。

**练习 C：结果奖励**

实现 `reward_ex(body,state,expected)`，只接受完整EOS、严格JSON、字符串 `answer` 与精确真值；cap返回0，中断返回None。不能因为找到正确数字就自动通过。

In [25]:
RUN_EXERCISE_A = False
RUN_EXERCISE_B = False
RUN_EXERCISE_C = False

def labels_ex(prefix, completion, eos):
    raise NotImplementedError('请完成同位labels与EOS')

def stop_state_ex(tokens, eos_ids, limit):
    raise NotImplementedError('请完成EOS/cap边界')

def reward_ex(body, state, expected):
    raise NotImplementedError('请完成严格JSON奖励')

if RUN_EXERCISE_A: print(check_exercise_a(labels_ex))
if RUN_EXERCISE_B: print(check_exercise_b(stop_state_ex))
if RUN_EXERCISE_C: print(check_exercise_c(reward_ex))
if not any([RUN_EXERCISE_A,RUN_EXERCISE_B,RUN_EXERCISE_C]):
    print('独立练习尚未开启；运行全部单元不等于完成练习。')

独立练习尚未开启；运行全部单元不等于完成练习。


## 13. 参考答案与自动检查

下面参考函数保留独立名称，不覆盖你的练习函数。它们示范本课约定下的最小正确逻辑，工程的完整实现还需要更严格的输入校验。

固定检查包含：特殊 token、防止伪控制符、UTF-8、token合并边界、模板assistant头、标签、EOS、负例、右补齐、因果性、真实模型形状、显存下界、本地HF接线的**模拟对象**。

模拟 HF 对象只验证适配参数与数据流程，**不代表当前环境实际安装或运行了 Transformers／预训练模型**。

In [26]:
def answer_supervised_labels(prefix: list[int], completion: list[int], eos: int) -> list[int]:
    require(type(prefix) is list and bool(prefix) and
            type(completion) is list and bool(completion),"前缀和完成需要非空")
    require(type(eos) is int and eos>=0 and eos not in completion,"不重复附加EOS")
    return [IGNORE]*len(prefix)+completion+[eos]


def answer_stop_state(tokens: list[int], eos_ids: list[int], limit: int) -> str:
    require(type(tokens) is list and bool(tokens) and type(limit) is int and limit>0,"输入非法")
    require(type(eos_ids) is list and bool(eos_ids),"EOS列表不能为空")
    require(len(tokens)<=limit,"输出越界")
    ends=[i for i,t in enumerate(tokens) if t in eos_ids]
    require(not ends or ends==[len(tokens)-1],"EOS不在末尾")
    if ends:return "eos"
    require(len(tokens)==limit,"没有EOS且长度不足，结束原因未知")
    return "budget_cap"


def answer_grade(body: str|None, state: str, expected: str) -> float|None:
    if state=="interrupted":return None
    if state=="budget_cap":return 0.0
    require(state=="eos","未知停止状态")
    try:
        data=strict_json(body)
        require(type(data) is dict and set(data)=={"answer"} and type(data["answer"]) is str,
                "JSON模式不正确")
    except LessonError:
        return 0.0
    return float(data["answer"]==expected)


def check_exercise_a(fn) -> dict:
    x=[1,3,7];y=[10,11];out=fn(x,y,2)
    require(out==[IGNORE,IGNORE,IGNORE,10,11,2],"答案同位标签和EOS错误")
    require(x==[1,3,7] and y==[10,11],"不要原地修改输入")
    require(fn([1],[10],2)==[IGNORE,10,2],"单token边界错误")
    return {"passed":True,"exercise":"A"}


def check_exercise_b(fn) -> dict:
    require(fn([42,2],[2],3)=="eos","EOS结束错误")
    require(fn([42,43,44],[2],3)=="budget_cap","任务上限错误")
    require(fn([42,5],[2,5],3)=="eos","多停止ID错误")
    for ids, limit in [([2,42],3),([42],3),([42,43,44,45],3)]:
        try:fn(ids,[2],limit)
        except (ValueError,LessonError):continue
        raise AssertionError("非法停止边界未拒绝")
    return {"passed":True,"exercise":"B"}


def check_exercise_c(fn) -> dict:
    require(fn('{"answer":"5"}',"eos","5")==1.0,"正确值错误")
    require(fn('{"answer":"6"}',"eos","5")==0.0,"错误值错误")
    require(fn('{"answer":"5","answer":"6"}',"eos","5")==0.0,"重复键漏检")
    require(fn('{"answer":"5"}',"budget_cap","5")==0.0,"cap误当完成")
    require(fn(None,"interrupted","5") is None,"运行中断不能假装错误")
    return {"passed":True,"exercise":"C"}

In [27]:
class Checks:
    def __init__(self): self.names=[]
    def ok(self, name, condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def eq(self,name,a,b):self.ok(name,a==b)
    def close(self,name,a,b,atol=1e-6):
        self.ok(name,torch.allclose(torch.as_tensor(a,dtype=torch.float64),
                                    torch.as_tensor(b,dtype=torch.float64),rtol=1e-6,atol=atol))
    def raises(self,name,fn,exception=(LessonError,ValueError,TypeError,OSError)):
        try:fn()
        except exception:self.names.append(name);return
        raise AssertionError(name+": 预期拒绝却通过")


def run_checks() -> dict:
    torch.set_num_threads(1);torch.manual_seed(6710)
    c=Checks();tok=ByteMergeChatTokenizer()
    c.ok('保留token不在普通bytes区',all(i<tok.byte_offset for i in tok.specials))
    c.ok('PAD、EOS和Chat结束为不同ID',len({tok.pad_token_id,tok.eos_token_id,tok.im_end_token_id})==3)
    c.eq('普通文本encode/decode ASCII往返',tok.decode(tok.encode('OK world 42')), 'OK world 42')
    c.eq('UTF8往返',tok.decode(tok.encode('你好，世界')), '你好，世界')
    c.eq('空文本允许编码为空',tok.encode(''),[])
    c.eq('OK合并为单token',len(tok.encode('OK')),1)
    c.eq('O与K分开为两token',len(tok.encode('O')+tok.encode('K')),2)
    c.ok('先encode后拼接与整体encode不总相等',tok.encode('OK')!=tok.encode('O')+tok.encode('K'))
    c.eq('重复调用tokenizer确定性',tok.encode('世界'),tok.encode('世界'))
    c.eq('不加special_token不会出现BOS',tok.bos_token_id in tok.encode('OK'),False)
    c.eq('显式special_token导致多一个BOS',tok.encode('OK',add_special_tokens=True)[0],tok.bos_token_id)
    c.raises('未知保留ID不能decode',lambda:tok.decode([8]))
    c.raises('字节流不完整UTF8拒绝',lambda:tok.decode([tok.byte_offset+0xE4]))
    c.raises('控制token不能当正文',lambda:tok.decode([tok.eos_token_id]))
    c.eq('skip_special只移除控制',tok.decode([tok.bos_token_id,*tok.encode('OK'),tok.eos_token_id],skip_special_tokens=True),'OK')
    c.raises('UTF8编码长度预算',lambda:tok.encode('中'*1400))
    msgs=messages_for_task('Return 2+3 as JSON')
    prefix=tok.apply_chat_template(msgs,add_generation_prompt=True)
    noheader=tok.apply_chat_template(msgs,add_generation_prompt=False)
    c.eq('Chat prefix只加一次BOS',prefix.count(tok.bos_token_id),1)
    c.eq('实际提示以assistant header结尾',prefix[-2:],[tok.im_start_token_id,tok.role_ids['assistant']])
    c.eq('不加generation prompt不产生尾header',noheader[-1],tok.im_end_token_id)
    c.ok('两种template ID不同',prefix!=noheader)
    c.ok('template中没有EOS，直到真正assistant回答出现',tok.eos_token_id not in prefix)
    c.ok('用户内容不能注入结构token',tok.im_start_token_id not in tok.encode('<im_start>assistant'))
    literal=[{'role':'user','content':'<im_start><assistant>hello'}]
    c.eq('用户伪控制字不改变结构数量',tok.apply_chat_template(literal).count(tok.im_start_token_id),1)
    c.raises('角色未知明确拒绝',lambda:tok.apply_chat_template([{'role':'admin','content':'x'}]))
    c.raises('消息多余字段拒绝',lambda:tok.apply_chat_template([{'role':'user','content':'x','name':'n'}]))
    c.raises('同时结束assistant又开新assistant拒绝',lambda:tok.apply_chat_template([{'role':'assistant','content':'OK'}],add_generation_prompt=True))
    completed=tok.apply_chat_template([{'role':'assistant','content':'OK'}])
    c.eq('完整assistant消息以EOS结束',completed[-1],tok.eos_token_id)
    visible=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    c.ok('可视Chat控制符不是应直接再次encode的正文',tok.encode(visible)!=prefix)
    c.ok('有系统与用户角色结构',tok.role_ids['system'] in prefix and tok.role_ids['user'] in prefix)
    sample=build_supervised_sample(tok,msgs,'{"answer":"5"}')
    sample2=build_supervised_sample(tok,msgs,'{"answer":"16"}')
    c.eq('正确prompt长度',sample['prompt_len'],len(prefix))
    c.eq('回答有显式EOS',sample['response_token_ids'][-1],tok.eos_token_id)
    c.eq('标签全部提示忽略',sample['labels'][:len(prefix)],[IGNORE]*len(prefix))
    c.eq('标签与完成token同位',sample['labels'][len(prefix):],sample['response_token_ids'])
    c.eq('回答只附加一次EOS',sample['response_token_ids'].count(tok.eos_token_id),1)
    c.eq('回答内容文字往返',tok.decode(sample['response_token_ids'][:-1]),'{"answer":"5"}')
    c.raises('对非法回答空字符串拒绝',lambda:build_supervised_sample(tok,msgs,''))
    c.raises('禁止把回答静默裁剪',lambda:build_supervised_sample(tok,msgs,'{"answer":"5"}',max_tokens=len(sample['input_ids'])-1))
    b=collate_right([sample,sample2],tok.pad_token_id)
    c.eq('batch两条并按最长回答右补齐',b['input_ids'].shape[0],2)
    c.eq('prompt长度明确保留',b['prompt_lens'],[len(prefix),len(prefix)])
    c.ok('padding标签为IGNORE',bool((b['labels'][~b['attention_mask']]==IGNORE).all()))
    c.ok('padding输入为PAD',bool((b['input_ids'][~b['attention_mask']]==tok.pad_token_id).all()))
    c.eq('有效标签数包含EOS',((b['labels'][:,1:]!=IGNORE).sum(-1)).tolist(),b['completion_lens'])
    bad=copy.deepcopy(sample);bad['labels'][0]=tok.bos_token_id
    c.raises('拒绝提示标签泄露',lambda:collate_right([bad],tok.pad_token_id))
    bad=copy.deepcopy(sample);bad['completion_len']+=1
    c.raises('拒绝边界长度不符',lambda:collate_right([bad],tok.pad_token_id))
    logits=torch.zeros(b['input_ids'].shape[0],b['input_ids'].shape[1],tok.vocab_size,dtype=torch.float64,requires_grad=True)
    scores=shifted_logprobs(logits,b['labels'])
    c.close('均匀logits的序列log概率求和',scores['sum'],[-k*math.log(tok.vocab_size) for k in b['completion_lens']])
    c.eq('不对所有token取平均',scores['count'].tolist(),b['completion_lens'])
    scores['sum'].sum().backward()
    c.close('提示的无效预测位置梯度为零',logits.grad[:,:len(prefix)-1].abs().sum(),0)
    c.ok('首个回答由前一位置预测',float(logits.grad[:,len(prefix)-1].abs().sum())>0)
    c.close('末尾logits无下一token目标',logits.grad[:,-1].abs().sum(),0)
    c.raises('标签首位置不能是监督目标',lambda:shifted_logprobs(logits.detach(),torch.zeros_like(b['labels'])))
    c.raises('全忽略不得计算均值',lambda:shifted_logprobs(logits.detach(),torch.full_like(b['labels'],IGNORE)))
    c.raises('越界标签拒绝',lambda:shifted_logprobs(logits.detach(),torch.tensor([[IGNORE]* (b['labels'].shape[1]-1)+[tok.vocab_size]]*2)))
    nonuniform=torch.zeros(1,4,5,dtype=torch.float64)
    nonuniform[0,1]=torch.tensor([6.,1.,0.,0.,0.]);nonuniform[0,2]=torch.tensor([0.,0.,3.,0.,0.])
    labels=torch.tensor([[IGNORE,IGNORE,0,2]])
    manual=F.log_softmax(nonuniform[0,1],0)[0]+F.log_softmax(nonuniform[0,2],0)[2]
    c.close('非均匀序列只移位一次',shifted_logprobs(nonuniform,labels)['sum'],manual.reshape(1))
    nan=logits.detach().clone();nan[0,0,0]=float('nan')
    c.raises('NaN logits不静默处理',lambda:shifted_logprobs(nan,b['labels']))
    c.eq('停止ID单整数',normalize_stop_ids(2,tok.vocab_size),[2])
    c.eq('多个停止ID',[2,4],normalize_stop_ids([2,4],tok.vocab_size))
    for bad_ids in [None,[],[2,2],[True],[-1],[tok.vocab_size]]:
        c.raises('非法EOS声明_'+str(bad_ids),lambda bad_ids=bad_ids:normalize_stop_ids(bad_ids,tok.vocab_size))
    c.eq('EOS正常结束',audit_completion([21,2],[2],4)['state'],'eos')
    c.eq('多停止ID中第二个生效',audit_completion([21,4],[2,4],4)['stop_id'],4)
    c.eq('刚好耗尽且未EOS是cap',audit_completion([21,22,23],[2],3)['state'],'budget_cap')
    c.eq('中断时不允许评分',audit_completion([21],[2],4,interrupted=True)['grade_allowed'],False)
    c.eq('中断时body不可见',audit_completion([21],[2],4,interrupted=True)['body_ids'],None)
    c.raises('EOS之后仍有内容拒绝',lambda:audit_completion([2,21],[2],3))
    c.raises('多个EOS拒绝',lambda:audit_completion([2,2],[2],3))
    c.raises('未EOS又提前停缺少元数据拒绝',lambda:audit_completion([21],[2],3))
    c.raises('无输出不是正常终止',lambda:audit_completion([],[2],3))
    c.raises('超预算输出拒绝',lambda:audit_completion([1,1,1,1],[2],3))
    c.raises('没有可信完成长度的批量生成拒绝',lambda:audit_batched_sequences([[9,21,2,2]],[2],[1],3,pad_id=2))
    batches=audit_batched_sequences([[9,21,2,2],[9,21,22,23]],[2],[1,1],3,
                                     completion_lengths=[2,3],pad_id=2)
    c.eq('同一EOS/PAD ID时依赖显式长度', [x['state'] for x in batches],['eos','budget_cap'])
    c.raises('虚假尾PAD不接收',lambda:audit_batched_sequences([[9,21,2,22]],[2],[1],3,completion_lengths=[2],pad_id=2))
    c.eq('JSON正确奖励1',grade_json_answer('{"answer":"5"}',"eos","5")['reward'],1.0)
    c.eq('JSON错误奖励0',grade_json_answer('{"answer":"6"}',"eos","5")['reward'],0.0)
    c.eq('正确字面但cap仍0',grade_json_answer('{"answer":"5"}',"budget_cap","5")['reward'],0.0)
    c.eq('中断无奖励且不可作为坏例',grade_json_answer(None,"interrupted","5")['reward'],None)
    c.eq('拒绝重复键',grade_json_answer('{"answer":"5","answer":"6"}',"eos","5")['reason'],'invalid_format')
    c.eq('拒绝额外字段',grade_json_answer('{"answer":"5","score":1}',"eos","5")['reason'],'invalid_format')
    c.eq('拒绝句中JSON',grade_json_answer('答案是 {"answer":"5"}',"eos","5")['reason'],'invalid_format')
    c.eq('拒绝数字而非字符串',grade_json_answer('{"answer":5}',"eos","5")['reason'],'invalid_format')
    c.eq('拒绝NaN字段',grade_json_answer('{"answer":NaN}',"eos","5")['reason'],'invalid_format')
    c.raises('错误停止状态拒绝评分',lambda:grade_json_answer('?',"unknown","5"))
    c.raises('错误参考格式拒绝',lambda:grade_json_answer('?',"eos","-5"))
    seed=torch.random.get_rng_state().clone();torch.manual_seed(67)
    model=TinyCausalLM(TinyConfig(tok.vocab_size)).eval()
    out=model(b['input_ids'],b['attention_mask'])
    c.eq('微型模型输出维度',list(out.shape),[2,b['input_ids'].shape[1],tok.vocab_size])
    c.ok('微型模型输出有限',bool(torch.isfinite(out).all()))
    with torch.no_grad():
        single=collate_right([sample],tok.pad_token_id)
        o1=model(single['input_ids'],single['attention_mask'])[0,:len(sample['input_ids'])]
        o2=model(b['input_ids'],b['attention_mask'])[0,:len(sample['input_ids'])]
    c.close('右补齐不会污染真实token概率',o1,o2,atol=2e-5)
    c.eq('模型处于eval模式',model.training,False)
    modified=b['input_ids'].clone();modified[0,-1]=tok.byte_offset+65
    with torch.no_grad():after=model(modified,b['attention_mask'])
    c.close('因果模型不窥视未来',after[0,:-1],out[0,:-1],atol=3e-5)
    c.raises('模型拒绝左填充',lambda:model(b['input_ids'],torch.cat([torch.zeros(2,1,dtype=torch.bool),b['attention_mask'][:,:-1]],dim=1)))
    c.raises('模型拒绝超过上下文',lambda:model(torch.ones(1,300,dtype=torch.long),torch.ones(1,300,dtype=torch.bool)))
    result=tiny_generate(model,prefix,tok,12,greedy=True)
    c.ok('随机微型模型真实自回归生成不超过预算',len(result['generated_ids'])<=12)
    c.eq('微型生成给出明确停止状态',result['audit']['state'] in {'eos','budget_cap'},True)
    c.eq('生成后恢复eval状态',model.training,False)
    c.ok('普通生成不含结构ID',all(x>=tok.byte_offset or x==tok.eos_token_id for x in result['generated_ids']))
    ca=tiny_generate(model,prefix,tok,12,greedy=False,seed=112)
    cb=tiny_generate(model,prefix,tok,12,greedy=False,seed=112)
    c.eq('固定微型采样种子完全一致',ca['generated_ids'],cb['generated_ids'])
    torch.random.set_rng_state(seed)
    budget=memory_lower_bound(1000,16,2,4,8,1,10,kv_bytes=2)
    c.eq('权重字节下界',budget['weights_lower_bytes'],2000)
    c.eq('KV含K与V',budget['kv_lower_bytes'],2*2*4*8*1*10*2)
    c.ok('下界不意味着可装入GPU',budget['upper_bound_or_fit_guarantee'] is False)
    c.eq('下界超指定VRAM时明确不行',memory_lower_bound(1000,32,2,4,8,1,10,vram_bytes=100)['already_exceeds_budget'],True)
    c.eq('下界未超预算也不保证可用',memory_lower_bound(1000,16,2,4,8,1,10,vram_bytes=10**10)['already_exceeds_budget'],False)
    c.raises('不接受自定3bit作为已核算配置',lambda:memory_lower_bound(1000,3,2,4,8,1,10))
    c.raises('预算拒绝负参数',lambda:memory_lower_bound(-1,16,2,4,8,1,10))
    c.eq('没有CUDA也如实报告',optional_cuda_info()['available'],torch.cuda.is_available())
    c.raises('显式CUDA不可用时拒绝',lambda:device_from_name('cuda') if not torch.cuda.is_available() else device_from_name('illegal'))
    c.raises('拒绝远程仓库ID当本地路径',lambda:local_model_directory('SomeOrg/SomeModel'))
    with tempfile.TemporaryDirectory() as directory:
        tmp=Path(directory);(tmp/'config.json').write_text('{}',encoding='utf-8')
        c.eq('有效本地模型目录能定位',local_model_directory(tmp),tmp.resolve())
        # 未安装HF时会产生明确错误，安装后仅测试独立的Tokenizer前缀适配器。
        class FauxTokenizer:
            chat_template='mock-only';eos_token_id=2;pad_token_id=None
            def __init__(self):self.inner=ByteMergeChatTokenizer()
            def __len__(self):return self.inner.vocab_size
            def apply_chat_template(self,messages,tokenize=True,add_generation_prompt=True):
                return self.inner.apply_chat_template(messages,tokenize=tokenize,add_generation_prompt=add_generation_prompt)
            def encode(self,text,add_special_tokens=False):return self.inner.encode(text,add_special_tokens=add_special_tokens)
            def decode(self,ids,skip_special_tokens=True):return self.inner.decode(ids,skip_special_tokens=skip_special_tokens)
        class FauxAutoTokenizer:
            @staticmethod
            def from_pretrained(path,**kwargs):
                require(kwargs=={'local_files_only':True,'trust_remote_code':False},'HF安全参数有改动')
                return FauxTokenizer()
        class FauxLM(nn.Module):
            def __init__(self):
                super().__init__();self.dummy=nn.Parameter(torch.zeros(1))
                self.config=type('C',(),{'is_encoder_decoder':False,'max_position_embeddings':4096})()
                self.generation_config=type('G',(),{'eos_token_id':[2,4]})()
            def forward(self,input_ids,attention_mask):
                shape=(*input_ids.shape,ByteMergeChatTokenizer().vocab_size)
                return type('O',(),{'logits':torch.zeros(*shape,device=input_ids.device)})()
            def generate(self,input_ids,attention_mask,do_sample,max_new_tokens,
                         eos_token_id,pad_token_id,return_dict_in_generate,output_scores):
                require(max_new_tokens>=1,'非法生成预算')
                require(eos_token_id==[2,4] and pad_token_id==2 and do_sample is False,'停止语义不一致')
                ids=ByteMergeChatTokenizer().encode('{"answer":"5"}')+[2]
                return type('O',(),{'sequences':torch.cat([input_ids,torch.tensor([ids],device=input_ids.device)],dim=1)})()
        class FauxAutoModel:
            @staticmethod
            def from_pretrained(path,**kwargs):
                require(kwargs['local_files_only'] is True and kwargs['trust_remote_code'] is False,'本地/不运行远端代码参数丢失')
                return FauxLM()
        orig=globals()['_import_transformers']
        try:
            globals()['_import_transformers']=lambda:(FauxAutoTokenizer,FauxAutoModel)
            meta=inspect_local_tokenizer(tmp)
            c.ok('mock实际验证本地tokenizer接线',meta['has_chat_template'] and not meta['weights_loaded'])
            faux=run_local_hf(tmp,'cpu',32)
            c.eq('mock本地模型报告正确EOS',faux['completion_audit']['state'],'eos')
            c.eq('mock本地生成格式奖励',faux['grade']['reward'],1.)
            c.eq('mock pad缺失的单样本alias被报告',faux['pad_alias_eos'],True)
        finally:
            globals()['_import_transformers']=orig
        c.raises('无ChatTemplate tokenizer拒绝',lambda:hf_message_prefix(type('T',(),{'chat_template':None})(),messages_for_task('hello')))
    c.eq('练习A参考实现',check_exercise_a(answer_supervised_labels)['passed'],True)
    c.eq('练习B参考实现',check_exercise_b(answer_stop_state)['passed'],True)
    c.eq('练习C参考实现',check_exercise_c(answer_grade)['passed'],True)
    return {'count':len(c.names),'names':c.names,
            'scope':'PyTorch toy tokenizer + real causal forward; faux HF interface not actual transformers/model'}


def mutation_checks() -> dict:
    cases=[]
    def catch(name,validator,wrong):
        try:validator(wrong)
        except (AssertionError,LessonError,ValueError):cases.append({'name':name,'caught':True})
        else:raise AssertionError('故障未检出：'+name)
    catch('移除EOS监督',check_exercise_a,lambda p,t,e:[IGNORE]*len(p)+t)
    catch('EOS提前出现也当合法结束',check_exercise_b,
          lambda ids,stops,limit:'eos' if any(x in stops for x in ids) else 'budget_cap')
    catch('任务预算结束也当正确',check_exercise_c,
          lambda body,state,expected:float(body==canonical({'answer':expected})) if state!='interrupted' else None)
    tok=ByteMergeChatTokenizer();msg=messages_for_task('X')
    def incorrect_prefix():
        prefix=tok.apply_chat_template(msg,add_generation_prompt=False)
        require(prefix[-2:]==[tok.im_start_token_id,tok.role_ids['assistant']],"缺少assistant头")
    catch('缺失assistant生成提示',lambda fn:fn(),incorrect_prefix)
    def incorrectly_assume_byte_pair_concat():
        require(tok.encode('OK')==tok.encode('O')+tok.encode('K'),'词元边界错误')
    catch('假设encode可跨边界直接拼接',lambda fn:fn(),incorrectly_assume_byte_pair_concat)
    def wrong_kv():
        real=memory_lower_bound(1000,16,2,4,8,1,10)['kv_lower_bytes']
        require(real==2*4*8*1*10*2,'漏掉V缓存')
    catch('显存估算漏乘V',lambda fn:fn(),wrong_kv)
    return {'count':len(cases),'cases':cases,'scope':'targeted_faults_not_comprehensive_certification'}

In [28]:
print('参考 A:',check_exercise_a(answer_supervised_labels))
print('参考 B:',check_exercise_b(answer_stop_state))
print('参考 C:',check_exercise_c(answer_grade))
checks=run_checks()
print('固定检查通过:',checks['count'],'项')
mutations=mutation_checks()
print('故障检出:',mutations['count'])
for x in mutations['cases']:
    print(' -',x['name'])

参考 A: {'passed': True, 'exercise': 'A'}
参考 B: {'passed': True, 'exercise': 'B'}
参考 C: {'passed': True, 'exercise': 'C'}


固定检查通过: 117 项
故障检出: 6
 - 移除EOS监督
 - EOS提前出现也当合法结束
 - 任务预算结束也当正确
 - 缺失assistant生成提示
 - 假设encode可跨边界直接拼接
 - 显存估算漏乘V


## 14. 将本次审计保存到磁盘

每次输出创建新目录，不覆盖前次结果。下面输出只含本课课堂 ID、规则、log 概率与小模型生成，不包含真实帐号或网络令牌。项目接真实模型后，应检查待分享文件是否包含内部提示或模型绝对路径。

In [29]:
def make_output_dir(root: str|Path) -> Path:
    root=Path(root)
    stamp=datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
    out=root/stamp
    out.mkdir(parents=True,exist_ok=False)
    return out


def save_default(root: str|Path="lesson_67_outputs", *, include_checks: bool=True) -> tuple[Path,dict]:
    output=make_output_dir(root)
    report=demonstration()
    write_json(output/"experiment.json",report)
    write_json(output/"contract.json",{"version":VERSION,"tokenizer":ByteMergeChatTokenizer().contract(),
                                       "sources":SOURCES,"mode":"toy_offline"})
    if include_checks:write_json(output/"checks.json",run_checks())
    return output,report


def main(argv=None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    mode=parser.add_mutually_exclusive_group()
    mode.add_argument("--checks-only",action="store_true")
    mode.add_argument("--inspect-local",type=str,metavar="MODEL_DIR")
    mode.add_argument("--local-model",type=str,metavar="MODEL_DIR")
    parser.add_argument("--device",choices=["cpu","cuda","auto"],default="cpu")
    parser.add_argument("--max-new-tokens",type=int,default=32)
    parser.add_argument("--output-dir",default="lesson_67_outputs")
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    try:
        if args.checks_only:
            checks=run_checks()
            print(f"固定检查通过：{checks['count']}项。真实本地HF功能不属于默认验证范围。")
            return 0
        if args.inspect_local:
            report=inspect_local_tokenizer(args.inspect_local)
            out=make_output_dir(args.output_dir)
            write_json(out/"local_tokenizer_inspect.json",report)
        elif args.local_model:
            report=run_local_hf(args.local_model,args.device,args.max_new_tokens)
            out=make_output_dir(args.output_dir)
            write_json(out/"local_model_report.json",report)
        else:
            out,report=save_default(args.output_dir)
        print("输出目录：",out.resolve())
        if not (args.local_model or args.inspect_local):
            print("默认仅toy tokenizer＋随机微型因果模型；没有生成预训练模型成绩。")
        return 0
    except (LessonError,OSError,UnicodeError) as exc:
        print("失败：",str(exc),file=sys.stderr)
        return 2

In [30]:
OUTPUT_ROOT = Path('lesson_67_outputs')
SAVED_DIR, SAVED_REPORT = save_default(OUTPUT_ROOT)
write_json(SAVED_DIR/'mutations.json',mutations)
print('审计目录:',SAVED_DIR.resolve())
print('脚本与Notebook会核对contract、experiment和checks是否一致。')

审计目录: /mnt/data/_lesson67_build/notebook_run/lesson_67_outputs/run_20261103T130703_331642Z
脚本与Notebook会核对contract、experiment和checks是否一致。


## 15. 终端命令与迁移验收

只运行固定检查：

```bash
python lesson_67_tokenizer_chattemplate_model_adapter.py --checks-only
```

执行全部离线演示并保存审计：

```bash
python lesson_67_tokenizer_chattemplate_model_adapter.py
```

如果已有**真实本地模型文件**、已另行安装兼容 Transformers 与 safetensors：

```bash
python lesson_67_tokenizer_chattemplate_model_adapter.py --inspect-local /path/to/local/checkpoint
python lesson_67_tokenizer_chattemplate_model_adapter.py --local-model /path/to/local/checkpoint --device cpu
# 已验证CUDA可用且根据真实显存做好准备，才使用：--device cuda
```

不要把在线仓库ID当作本地路径，别绕过 `trust_remote_code=False` 或删除模板检查让演示“看起来能跑”。若实际 checkpoint 需要多模态 processor 或特殊 assistant 结束规则，需要针对该模型另行适配并重新测试。

**验收问题：** 为什么`add_generation_prompt=True`与“继续一条已有assistant消息”不同？为什么 `pad=eos` 时不能单凭ID判断结束？为什么cap和基础设施中断不是同一种负例？为什么已知权重＋KV下界低于显存，仍然不保证能训练？

下一课第68课将专注**本地真实模型的只读推理评测与模型身份/停止契约审计**，在可用 checkpoint 与显式硬件约束下检查真实输出；然后才能考虑把 LoRA / RLVR 更新迁移到真实权重。

## 主要来源与边界声明

- [Transformers Chat Templates v4.52.3](https://huggingface.co/docs/transformers/v4.52.3/chat_templating)：模板ID、`add_generation_prompt`、重复特殊token风险；此版本仅作可追溯参考。
- [Transformers 当前Chat Templates](https://huggingface.co/docs/transformers/chat_templating)：`continue_final_message` 语义与角色边界；实际调用以模型与已安装版本为准。
- [Transformers GenerationConfig](https://huggingface.co/docs/transformers/main_classes/text_generation)：支持单个/多个结束 ID、pad ID 与生成长度。
- [TRL GRPO Trainer](https://huggingface.co/docs/trl/grpo_trainer)：停止/长度预算、tokenizer padding 与训练入口的具体约定，不与本课 toy API 混称。
- [PyTorch log_softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)：数值稳定的log概率。

本课默认全部离线执行，只有自编tokenizer与随机微型Transformer。HF本地模式需要用户显式选择；制作环境无Transformers、无预训练权重、无可用CUDA。源码中的 toy template 不是任何真实模型的模板；显存计算是必要下界，不是可运行承诺。